Structured output

Pydantic

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


model = init_chat_model("groq:openai/gpt-oss-120b")




In [21]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="the title of the movie")
    year:int= Field(description="the year movie got released")
    director:str = Field(description="the director of the movie")
    rating:int = Field(description="rating of the movie")

model_with_structure = model.with_structured_output(Movie)
model_with_structure

responces = model_with_structure.invoke("provide details about the movie the avengers")
print(responces)

title='The Avengers' year=2012 director='Joss Whedon' rating=8


Message output alongside parsed structure

In [19]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(..., description="the title of the movie")
    year:int= Field(..., description="the year movie got released")
    director:str = Field(..., description="the director of the movie")
    rating:int = Field(..., description="rating of the movie")


model_with_structure = model.with_structured_output(Movie,include_raw=True)


responce = model_with_structure.invoke("provide deteails aboout the movie the avengers")
responce


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "provide details about the movie the avengers". Likely they want info like director, rating, title, year. Use the function Movie. Provide details. Let\'s call function with appropriate data. The Avengers (2012) directed by Joss Whedon, rating maybe 8.0 (IMDb). Title "The Avengers". Year 2012. Use function.', 'tool_calls': [{'id': 'fc_7db83721-8b9d-4c73-9e48-f779b59f145a', 'function': {'arguments': '{"director":"Joss Whedon","rating":8,"title":"The Avengers","year":2012}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 134, 'prompt_tokens': 157, 'total_tokens': 291, 'completion_time': 0.282263777, 'completion_tokens_details': {'reasoning_tokens': 79}, 'prompt_time': 0.089287541, 'prompt_tokens_details': None, 'queue_time': 0.324520261, 'total_time': 0.371551318}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_854fa9be4c', 'service_tier': 'on

Nested structure

In [18]:
from pydantic import BaseModel,Field

class Actors(BaseModel):
    name :str
    role :str


class Movie(BaseModel):
    title:str=Field(description="the title of the movie")
    year:int= Field(description="the year movie got released")
    cast :list[Actors]
    genres :list[str]
    director:str = Field(description="the director of the movie")
    rating:int = Field(description="rating of the movie")


model_with_structure = model.with_structured_output(Movie)


responce = model_with_structure.invoke("provide details about the movie the avengers")
responce


Movie(title='The Avengers', year=2012, cast=[Actors(name='Robert Downey Jr.', role='Tony Stark / Iron Man'), Actors(name='Chris Evans', role='Steve Rogers / Captain America'), Actors(name='Mark Ruffalo', role='Bruce Banner / Hulk'), Actors(name='Chris Hemsworth', role='Thor'), Actors(name='Scarlett Johansson', role='Natasha Romanoff / Black Widow'), Actors(name='Jeremy Renner', role='Clint Barton / Hawkeye'), Actors(name='Tom Hiddleston', role='Loki'), Actors(name='Clark Gregg', role='Phil Coulson'), Actors(name='Samuel L. Jackson', role='Nick Fury')], genres=['Action', 'Adventure', 'Sci-Fi', 'Superhero'], director='Joss Whedon', rating=8)

Typedict

In [20]:
from typing_extensions import TypedDict,Annotated


class Movie(TypedDict):
    title:Annotated [str,..., "the title of the movie"]
    year:Annotated [int, ...,"the year movie got released"]
    director:Annotated [str, ..., "the director of the movie"]
    rating:Annotated[int ,...,"rating of the movie"]

model_with_structure = model.with_structured_output(Movie)
model_with_structure


responces = model_with_structure.invoke("provide details about the movie the avengers")
print(responces)

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}


Agents


In [22]:
from pydantic import BaseModel,Field

from langchain.agents import create_agent

class Movie(BaseModel):
    title:str=Field(description="the title of the movie")
    year:int= Field(description="the year movie got released")
    director:str = Field(description="the director of the movie")
    rating:int = Field(description="rating of the movie")


model = init_chat_model(
    "groq:openai/gpt-oss-120b",
    temperature=0
)
agent = create_agent(
    model = model,
    response_format=Movie

)

result = agent.invoke ({
    "messages":[{"role":"user","content":"provide movie info for avengers movie"}]

})

result




{'messages': [HumanMessage(content='provide movie info for avengers movie', additional_kwargs={}, response_metadata={}, id='d668262d-e39b-4d95-8941-200c660b2a2a'),
  AIMessage(content='{"title":"The Avengers","year":2012,"director":"Joss Whedon","rating":8}', additional_kwargs={'reasoning_content': 'The user asks: "provide movie info for avengers movie". We need to output JSON matching the Movie schema. Must include fields: title, year, director, rating. Provide info for "The Avengers" (2012) directed by Joss Whedon, rating maybe 8 (or integer). Provide rating as integer. Provide year 2012. Title "The Avengers". Provide director "Joss Whedon". Provide rating maybe 8. Ensure JSON compact, no extra whitespace, all fields present. Output only JSON.'}, response_metadata={'token_usage': {'completion_tokens': 145, 'prompt_tokens': 233, 'total_tokens': 378, 'completion_time': 0.307218684, 'completion_tokens_details': {'reasoning_tokens': 110}, 'prompt_time': 0.009947668, 'prompt_tokens_detail

Dataclass

In [23]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain_groq import data

@dataclass
class contactinfo:
    name :str #the name of the person
    email : str #tha mail of the person



model = init_chat_model(
    "groq:openai/gpt-oss-120b",
    temperature=0
)
agent = create_agent(
    model = model,
    response_format=contactinfo

)

result = agent.invoke ({
    "messages":[{"role":"user","content":"extract info from :raj, raj@gmail.com"}]

})

result


{'messages': [HumanMessage(content='extract info from :raj, raj@gmail.com', additional_kwargs={}, response_metadata={}, id='92e72ee9-a579-488b-b1ec-e26bd002a5df'),
  AIMessage(content='{"name":"raj","email":"raj@gmail.com"}', additional_kwargs={'reasoning_content': 'The user says: "extract info from :raj, raj@gmail.com". The system wants a JSON object according to the schema "contactinfo". The schema requires fields: name (string) and email (string). So we need to output JSON with name "raj" and email "raj@gmail.com". Must be compact JSON, parseable, include all fields. Output only the JSON object.\n\nThus final answer: {"name":"raj","email":"raj@gmail.com"}'}, response_metadata={'token_usage': {'completion_tokens': 119, 'prompt_tokens': 180, 'total_tokens': 299, 'completion_time': 0.246415536, 'completion_tokens_details': {'reasoning_tokens': 95}, 'prompt_time': 0.009037689, 'prompt_tokens_details': None, 'queue_time': 0.215137057, 'total_time': 0.255453225}, 'model_name': 'openai/gpt